# Duplicates and data quality

IMP-04. Checks duplicates, value types, codes, plausibility ranges, derived columns and follicle count vs size. Each problem is added to a list with a proposed action; nothing in the data is changed here.

In [1]:
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "PCOS_data_without_infertility.xlsx"
pd.set_option("display.max_rows", 60)

df = pd.read_excel(DATA_FILE, sheet_name="Full_new")
df.columns = df.columns.str.strip()          # remove hidden leading/trailing spaces (found in IMP-02)

LABEL = "PCOS (Y/N)"
ID = "Patient File No."
ID_COLS = ["Sl. No", ID]
STRAY = [c for c in df.columns if c.startswith("Unnamed")]
FEATURES = [c for c in df.columns if c not in ID_COLS + [LABEL] + STRAY]
num = df[FEATURES].apply(pd.to_numeric, errors="coerce")   # numeric view; text entries become blank

problems = []   # every problem found is added here, for the summary at the end

def add(check, idx, column, value, action, reason):
    problems.append({"check": check, "patient_file_no": int(df.at[idx, ID]), "column": column,
                     "value": value, "pcos": int(df.at[idx, LABEL]),
                     "proposed_action": action, "reason": reason})

print(f"Records: {len(df)} | features: {len(FEATURES)}")

Records: 541 | features: 41


## Duplicates

In [2]:
checks = {
    "Duplicate Sl. No": df["Sl. No"].duplicated(keep=False).sum(),
    "Duplicate Patient File No.": df[ID].duplicated(keep=False).sum(),
    "Identical rows (features + label, IDs ignored)": df[FEATURES + [LABEL]].duplicated(keep=False).sum(),
    "Identical features, different label": (
        df[df[FEATURES].duplicated(keep=False)].groupby(FEATURES, dropna=False)[LABEL].nunique().gt(1).sum()
    ),
}
pd.Series(checks, name="records involved").to_frame()

,records involved
Duplicate Sl. No,0
Duplicate Patient File No.,0
"Identical rows (features + label, IDs ignored)",0
"Identical features, different label",0


## Text in number columns

In [3]:
for col in FEATURES:
    if df[col].dtype == "object":
        for idx in df.index[num[col].isna() & df[col].notna()]:
            raw = str(df.at[idx, col])
            fixed = pd.to_numeric(raw.rstrip("."), errors="coerce")
            if pd.notna(fixed):
                add("Text in number column", idx, col, repr(raw), f"Correct to {fixed}", "Typo: trailing full stop")
            else:
                add("Text in number column", idx, col, repr(raw), "Set to missing", "Not a number; true value unknown")

pd.DataFrame([p for p in problems if p["check"] == "Text in number column"])

,check,patient_file_no,column,value,pcos,proposed_action,reason
0,Text in number column,124,II beta-HCG(mIU/mL),'1.99.',1,Correct to 1.99,Typo: trailing full stop
1,Text in number column,306,AMH(ng/mL),'a',0,Set to missing,Not a number; true value unknown


## Codes

Y/N columns should be 0/1, cycle 2 or 4, blood group 11-18 (from the Instructions sheet).

In [4]:
ALLOWED = {"Cycle(R/I)": {2, 4}, "Blood Group": set(range(11, 19))}
ALLOWED.update({c: {0, 1} for c in FEATURES if "(Y/N)" in c})

for col, allowed in ALLOWED.items():
    bad = num[col].notna() & ~num[col].isin(allowed)
    for idx in df.index[bad]:
        add("Undefined code", idx, col, num.at[idx, col], "Set to missing",
            f"Allowed codes are {sorted(allowed)}; intended value unknown")

print("Columns checked:", len(ALLOWED))
pd.DataFrame([p for p in problems if p["check"] == "Undefined code"])

Columns checked: 10


,check,patient_file_no,column,value,pcos,proposed_action,reason
0,Undefined code,513,Cycle(R/I),5,1,Set to missing,"Allowed codes are [2, 4]; intended value unknown"


## Values outside plausible ranges

The ranges are wide screening bounds, not clinical reference ranges, and still need checking against a cited source.

In [5]:
PLAUSIBLE = {   # column: (lowest possible, highest possible)
    "Age (yrs)": (12, 60), "Weight (Kg)": (25, 200), "Height(Cm)": (120, 200), "BMI": (10, 70),
    "Pulse rate(bpm)": (40, 150), "RR (breaths/min)": (8, 40), "Hb(g/dl)": (5, 20),
    "Cycle length(days)": (0, 15), "Marraige Status (Yrs)": (0, 45), "No. of aborptions": (0, 15),
    "FSH(mIU/mL)": (0, 200), "LH(mIU/mL)": (0, 200), "Hip(inch)": (20, 70), "Waist(inch)": (20, 70),
    "Waist:Hip Ratio": (0.5, 1.5), "TSH (mIU/L)": (0, 100), "AMH(ng/mL)": (0, 60), "PRL(ng/mL)": (0, 300),
    "Vit D3 (ng/mL)": (0, 150), "PRG(ng/mL)": (0, 100), "RBS(mg/dl)": (40, 600),
    "BP _Systolic (mmHg)": (70, 250), "BP _Diastolic (mmHg)": (40, 150),
    "Follicle No. (L)": (0, 60), "Follicle No. (R)": (0, 60),
    "Avg. F size (L) (mm)": (0, 40), "Avg. F size (R) (mm)": (0, 40), "Endometrium (mm)": (0, 30),
}
KEEP_IF_POSSIBLE = {"AMH(ng/mL)"}   # extreme but biologically possible in PCOS

for col, (low, high) in PLAUSIBLE.items():
    values = num[col]
    median = values.median()
    for idx in df.index[(values < low) | (values > high)]:
        v = values[idx]
        ratio = v / median if median else None
        times = "" if ratio is None else f"{ratio:.0f}x the median" if ratio >= 1 else f"{ratio:.2f}x the median"
        if col in KEEP_IF_POSSIBLE:
            add("Outside plausible range", idx, col, v, "Keep and flag",
                f"Extreme ({times}) but possible in PCOS; check influence later")
        else:
            add("Outside plausible range", idx, col, v, "Set to missing",
                f"Not physiologically possible (range {low}-{high}; {times}); true value unknown")

print("Columns checked:", len(PLAUSIBLE), "(beta-HCG has no upper bound: pregnancy values are legitimately huge)")
pd.DataFrame([p for p in problems if p["check"] == "Outside plausible range"])

Columns checked: 28 (beta-HCG has no upper bound: pregnancy values are legitimately huge)


,check,patient_file_no,column,value,pcos,proposed_action,reason
0,Outside plausible range,224,Pulse rate(bpm),18.00,0,Set to missing,Not physiologically possible (range 40-150; 0....
1,Outside plausible range,297,Pulse rate(bpm),13.00,0,Set to missing,Not physiologically possible (range 40-150; 0....
2,Outside plausible range,330,FSH(mIU/mL),5052.00,0,Set to missing,Not physiologically possible (range 0-200; 104...
3,Outside plausible range,456,LH(mIU/mL),2018.00,1,Set to missing,Not physiologically possible (range 0-200; 877...
4,Outside plausible range,268,AMH(ng/mL),66.00,1,Keep and flag,Extreme (18x the median) but possible in PCOS;...
5,Outside plausible range,192,Vit D3 (ng/mL),6014.66,1,Set to missing,Not physiologically possible (range 0-150; 232...
6,Outside plausible range,196,Vit D3 (ng/mL),5418.60,1,Set to missing,Not physiologically possible (range 0-150; 209...
7,Outside plausible range,162,BP _Systolic (mmHg),12.00,0,Set to missing,Not physiologically possible (range 70-250; 0....
8,Outside plausible range,201,BP _Diastolic (mmHg),8.00,0,Set to missing,Not physiologically possible (range 40-150; 0....


## Statistical outliers (3 x IQR)

Context only; nothing is added to the problem list from this. Hormones are skewed, and for mostly-zero columns like abortions the IQR is 0.

In [6]:
rows = []
for col in FEATURES:
    values = num[col].dropna()
    if values.nunique() <= 2:          # skip yes/no columns
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    extreme = values[(values < q1 - 3 * iqr) | (values > q3 + 3 * iqr)]
    if len(extreme):
        rows.append({"column": col, "median": values.median(), "min": values.min(), "max": values.max(),
                     "extreme_values": len(extreme)})
pd.DataFrame(rows).sort_values("extreme_values", ascending=False).reset_index(drop=True)

,column,median,min,max,extreme_values
0,No. of aborptions,0.000000,0.000000,5.000000,104
1,II beta-HCG(mIU/mL),1.990000,0.990000,25000.000000,56
2,FSH/LH,2.169231,0.002146,1372.826087,29
3,I beta-HCG(mIU/mL),20.000000,1.300000,32460.970000,27
4,Cycle length(days),5.000000,0.000000,12.000000,18
5,TSH (mIU/L),2.260000,0.040000,65.000000,13
6,PRG(ng/mL),0.320000,0.047000,85.000000,10
7,AMH(ng/mL),3.700000,0.100000,66.000000,9
8,PRL(ng/mL),21.920000,0.400000,128.240000,8
9,Pulse rate(bpm),72.000000,13.000000,82.000000,5


## Derived columns

Recorded BMI, waist:hip and FSH/LH compared with values recalculated from their inputs (flagged if more than 5% apart).

In [7]:
DERIVED = {
    "BMI": num["Weight (Kg)"] / (num["Height(Cm)"] / 100) ** 2,
    "Waist:Hip Ratio": num["Waist(inch)"] / num["Hip(inch)"],
    "FSH/LH": num["FSH(mIU/mL)"] / num["LH(mIU/mL)"],
}
for col, recalculated in DERIVED.items():
    gap = (num[col] - recalculated).abs() / recalculated.abs()
    for idx in df.index[gap > 0.05]:
        add("Derived value mismatch", idx, col, num.at[idx, col], f"Recompute ({recalculated[idx]:.2f})",
            "Recorded value disagrees with its inputs, which are plausible")

# FSH/LH inherits errors from FSH or LH even when the arithmetic is correct
bad_inputs = {p["patient_file_no"] for p in problems
              if p["column"] in ("FSH(mIU/mL)", "LH(mIU/mL)") and p["proposed_action"] == "Set to missing"}
for idx in df.index[df[ID].isin(bad_inputs)]:
    add("Derived from an error", idx, "FSH/LH", num.at[idx, "FSH/LH"], "Set to missing",
        "Calculated from an FSH or LH value that is itself an error")

pd.DataFrame([p for p in problems if p["check"] in ("Derived value mismatch", "Derived from an error")])

,check,patient_file_no,column,value,pcos,proposed_action,reason
0,Derived value mismatch,441,BMI,20.300000,0,Recompute (21.87),"Recorded value disagrees with its inputs, whic..."
1,Derived from an error,330,FSH/LH,1372.826087,0,Set to missing,Calculated from an FSH or LH value that is its...
2,Derived from an error,456,FSH/LH,0.002146,1,Set to missing,Calculated from an FSH or LH value that is its...


Formula cells vs typed cells in the derived columns.

In [8]:
ws = load_workbook(DATA_FILE)["Full_new"]
header = [str(c.value).strip() for c in ws[1]]
formula_cells = {}
for row in ws.iter_rows(min_row=2):
    for cell in row:
        if isinstance(cell.value, str) and cell.value.startswith("="):
            col = header[cell.column - 1]
            formula_cells[col] = formula_cells.get(col, 0) + 1

pd.DataFrame({"formula_cells": formula_cells, "typed_cells": {c: len(df) - n for c, n in formula_cells.items()}})

,formula_cells,typed_cells
FSH/LH,532,9
Waist:Hip Ratio,532,9
BMI,299,242


## Follicle count vs size

A count of 0 with an average size above 0 is contradictory.

In [9]:
for side in ["L", "R"]:
    count_col, size_col = f"Follicle No. ({side})", f"Avg. F size ({side}) (mm)"
    contradiction = (num[count_col] == 0) & (num[size_col] > 0)
    for idx in df.index[contradiction]:
        add("Contradiction", idx, f"{count_col} / {size_col}",
            f"count 0, size {num.at[idx, size_col]:g}", "Review",
            "Either the count or the size is wrong; cannot tell which from the data")

pd.DataFrame([p for p in problems if p["check"] == "Contradiction"])

,check,patient_file_no,column,value,pcos,proposed_action,reason
0,Contradiction,129,Follicle No. (L) / Avg. F size (L) (mm),"count 0, size 15",0,Review,Either the count or the size is wrong; cannot ...
1,Contradiction,236,Follicle No. (L) / Avg. F size (L) (mm),"count 0, size 17",0,Review,Either the count or the size is wrong; cannot ...
2,Contradiction,252,Follicle No. (L) / Avg. F size (L) (mm),"count 0, size 17",0,Review,Either the count or the size is wrong; cannot ...
3,Contradiction,128,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 15",0,Review,Either the count or the size is wrong; cannot ...
4,Contradiction,262,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 20",0,Review,Either the count or the size is wrong; cannot ...
5,Contradiction,289,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 18",0,Review,Either the count or the size is wrong; cannot ...
6,Contradiction,299,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 12",0,Review,Either the count or the size is wrong; cannot ...
7,Contradiction,300,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 18",0,Review,Either the count or the size is wrong; cannot ...
8,Contradiction,410,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 11",0,Review,Either the count or the size is wrong; cannot ...
9,Contradiction,529,Follicle No. (R) / Avg. F size (R) (mm),"count 0, size 14",0,Review,Either the count or the size is wrong; cannot ...


## Problem list

In [10]:
register = pd.DataFrame(problems)
display(register.groupby(["check", "proposed_action"]).size().to_frame("cells"))
print(f"Problems found: {len(register)} cells in {register['patient_file_no'].nunique()} records "
      f"({100 * register['patient_file_no'].nunique() / len(df):.1f}% of records)")
register

cells
check                   proposed_action         
Contradiction           Review                13
Derived from an error   Set to missing         2
Derived value mismatch  Recompute (21.87)      1
Outside plausible range Keep and flag          1
                        Set to missing         8
Text in number column   Correct to 1.99        1
                        Set to missing         1
Undefined code          Set to missing         1

Problems found: 28 cells in 26 records (4.8% of records)


,check,patient_file_no,column,value,pcos,proposed_action,reason
0,Text in number column,124,II beta-HCG(mIU/mL),'1.99.',1,Correct to 1.99,Typo: trailing full stop
1,Text in number column,306,AMH(ng/mL),'a',0,Set to missing,Not a number; true value unknown
2,Undefined code,513,Cycle(R/I),5,1,Set to missing,"Allowed codes are [2, 4]; intended value unknown"
3,Outside plausible range,224,Pulse rate(bpm),18,0,Set to missing,Not physiologically possible (range 40-150; 0....
4,Outside plausible range,297,Pulse rate(bpm),13,0,Set to missing,Not physiologically possible (range 40-150; 0....
5,Outside plausible range,330,FSH(mIU/mL),5052.0,0,Set to missing,Not physiologically possible (range 0-200; 104...
6,Outside plausible range,456,LH(mIU/mL),2018.0,1,Set to missing,Not physiologically possible (range 0-200; 877...
7,Outside plausible range,268,AMH(ng/mL),66.0,1,Keep and flag,Extreme (18x the median) but possible in PCOS;...
8,Outside plausible range,192,Vit D3 (ng/mL),6014.66,1,Set to missing,Not physiologically possible (range 0-150; 232...
9,Outside plausible range,196,Vit D3 (ng/mL),5418.6,1,Set to missing,Not physiologically possible (range 0-150; 209...


## Observations

- There are no duplicate patients or rows, so duplicates won't cause leakage between training and test data.
- 8 values are clearly wrong: pulse of 13 and 18, blood pressure of 12 and 8, FSH 5052, LH 2018, and vitamin D3 of 6014.66 and 5418.6. These look like typing or decimal-point mistakes.
- AMH of 66 (patient 268) is very high, but it can happen in PCOS, so I'd keep it.
- Patient 513 has a cycle code of 5, but only 2 (regular) and 4 (irregular) are used.
- Patient 441's BMI (20.3) doesn't match their weight and height; it should be about 21.9. Most BMI values are Excel formulas, but this one was typed by hand.
- The FSH/LH ratios for patients 330 and 456 were calculated from the wrong FSH and LH values, so they're wrong too.
- 13 records have a follicle count of 0 but a follicle size above 0, which doesn't make sense. All 13 are non-PCOS patients, and 10 are for the right ovary. This matters because follicle count is one of the strongest predictors.
- The IQR check flags a lot of values, but most are normal, because hormone levels are naturally skewed. So I used plausible ranges to decide what's actually wrong. I still need to check these ranges against a proper clinical source.
- In total there are 28 problem values in 26 records (4.8% of records). I can't check whether the values are actually true without the hospital records, which is a limitation.